In [15]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load the original dataset
# --------------------------------------------------
file_path = r"Michigan all Data.csv"
df = pd.read_csv(file_path, low_memory=False)

# Convert wave identifier to numeric
df["YYYYMM"] = pd.to_numeric(df["YYYYMM"], errors="coerce")

# --------------------------------------------------
# 2. Select December 2021
# --------------------------------------------------
target_wave = 202112

wave = df.loc[df["YYYYMM"] == target_wave].copy()

print("Selected wave: December 2021")
print("Number of respondents:", len(wave))

if wave.empty:
    raise ValueError(
        "No December 2021 records found. Check the YYYYMM column."
    )

# --------------------------------------------------
# 3. Convert selected fields to numeric safely
# --------------------------------------------------
variables = [
    "ID", "AGE", "BIRTHM", "BIRTHY", "EDUCATION", "SEX", "MARRY",
    "NUMKID", "NUMADT", "INCOME", "HOMEOWN", "WT"
]

for col in variables:
    wave[col] = pd.to_numeric(
        wave[col].astype(str).str.strip(),
        errors="coerce"
    )



Selected wave: December 2021
Number of respondents: 603


In [12]:
# --------------------------------------------------
# 4. Define categorical labels
# --------------------------------------------------

labels = {
    "SEX": {
        1: "Male",
        2: "Female"
    },

    "EDUCATION": {
        1: "Grade 1–12, no diploma",
        2: "High school diploma or GED",
        3: "Some college, no degree",
        4: "Associate degree or vocational certification",
        5: "Bachelor's degree",
        6: "Master's degree",
        7: "Professional degree",
        8: "Doctorate",
        99: "NA"
    },

    "MARRY": {
        1:	"Married/partner",
        2:	"Separated",
        3:	"Divorced",
        4:	"Widowed",
        5:	"Never married"
    },

    "HOMEOWN": {
        # Verify these codes against the December 2021 codebook
        1: "Owns or buying",
        2: "Rent",
        99: "NA"
    }
}

# --------------------------------------------------
# 5. Define the household income brackets
# --------------------------------------------------
income_brackets = [
    "Under $10,000",
    "$10,000–$14,999",
    "$15,000–$19,999",
    "$20,000–$24,999",
    "$25,000–$29,999",
    "$30,000–$34,999",
    "$35,000–$39,999",
    "$40,000–$44,999",
    "$45,000–$49,999",
    "$50,000–$59,999",
    "$60,000–$74,999",
    "$75,000–$99,999",
    "$100,000–$124,999",
    "$125,000–$149,999",
    "$150,000–$174,999",
    "$175,000–$199,999",
    "$200,000–$224,999",
    "$225,000–$249,999",
    "$250,000 and above"
]

# --------------------------------------------------
# 6. Check what INCOME actually contains
# --------------------------------------------------
print("\nINCOME values in December 2021:")
print(wave["INCOME"].value_counts(dropna=False).sort_index().head(30))

# IMPORTANT:
# Only use this mapping if INCOME contains category codes 1–19.
# If INCOME contains dollar amounts or midpoint-coded values,
# stop and verify the codebook before mapping it.

income_codes = set(wave["INCOME"].dropna().unique())
expected_codes = set(range(1, 20))

if income_codes.issubset(expected_codes):
    labels["INCOME"] = {
        i: bracket
        for i, bracket in enumerate(income_brackets, start=1)
    }

    print("\nINCOME appears compatible with category codes 1–19.")
else:
    print(
        "\nWARNING: INCOME does not contain only codes 1–19. "
        "Income labels have NOT been applied. "
        "Check the codebook before proceeding."
    )




INCOME values in December 2021:
INCOME
5000.0      3
5200.0      1
6360.0      1
9000.0      2
9600.0      1
10000.0     2
10400.0     1
11000.0     2
12000.0     2
12500.0     1
13000.0     2
14000.0     2
15000.0     4
15750.0     1
17500.0     1
18000.0     3
19221.0     1
20000.0     8
22000.0     2
23000.0     1
24000.0     2
25000.0     5
26000.0     3
26208.0     1
27000.0     2
27500.0     2
28000.0     4
28500.0     1
30000.0    14
32000.0     2
Name: count, dtype: int64



In [13]:
# --------------------------------------------------
# 7. Create a separate copy with only selected variables
# --------------------------------------------------


persona_df = wave[variables].copy()

# Apply verified categorical mappings
for col, mapping in labels.items():
    if col in persona_df.columns:
        persona_df[col] = persona_df[col].map(mapping)

# --------------------------------------------------
# 8. Rename columns for readability
# --------------------------------------------------
persona_df = persona_df.rename(columns={
    "ID": "Respondent_ID",
    "AGE": "Age",
    "BIRTHM": "Birth_Month",
    "BIRTHY": "Birth_Year",
    "EDUC": "Education",
    "SEX": "Gender",
    "MARRY": "Marital_Status",
    "NUMKID": "Number_of_Children",
    "NUMADT": "Number_of_Adults",
    "INCOME": "Household_Income",
    "HOMEOWN": "Homeownership",
    "WT": "Survey_Weight"
})

# --------------------------------------------------
# 9. Inspect the final dataset
# --------------------------------------------------
print("\nFinal persona dataset")
print("Number of respondents:", len(persona_df))
print("Number of columns:", len(persona_df.columns))
print("\nColumns:")
print(persona_df.columns.tolist())

print("\nFirst five rows:")
display(persona_df.head())

print("\nMissing values by column:")
print(persona_df.isna().sum())

# Optional: save the result
persona_df.to_csv(
    "Michigan_December_2021_persona_data.csv",
    index=False
)



Final persona dataset
Number of respondents: 603
Number of columns: 12

Columns:
['Respondent_ID', 'Age', 'Birth_Month', 'Birth_Year', 'EDUCATION', 'Gender', 'Marital_Status', 'Number_of_Children', 'Number_of_Adults', 'Household_Income', 'Homeownership', 'Survey_Weight']

First five rows:


,Respondent_ID,Age,Birth_Month,Birth_Year,EDUCATION,Gender,Marital_Status,Number_of_Children,Number_of_Adults,Household_Income,Homeownership,Survey_Weight
300703,1,73.0,4.0,1948.0,NaN,Male,Married/partner,0,2,58000.0,Owns or buying,0.75
300704,10,40.0,10.0,1981.0,NaN,Female,Married/partner,3,2,130000.0,Owns or buying,1.25
300705,100,25.0,5.0,1996.0,NaN,Female,Never married,2,4,NaN,Owns or buying,1.00
300706,1001,65.0,9.0,1956.0,NaN,Male,Divorced,0,1,26000.0,Owns or buying,1.25
300707,1002,73.0,8.0,1948.0,NaN,Male,Married/partner,0,2,87500.0,Owns or buying,0.50



Missing values by column:
Respondent_ID           0
Age                     6
Birth_Month            36
Birth_Year              6
EDUCATION             603
Gender                  0
Marital_Status          0
Number_of_Children      0
Number_of_Adults        0
Household_Income       33
Homeownership           0
Survey_Weight           0
dtype: int64


In [6]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load data and select the latest available wave
# --------------------------------------------------
file_path = r"Michigan data all.csv"
df = pd.read_csv(file_path, low_memory=False)

df["YYYYMM"] = pd.to_numeric(df["YYYYMM"], errors="coerce")

target_wave = 202112
wave = df.loc[df["YYYYMM"] == target_wave].copy()

print("Selected wave: December 2021")

print("Records:", len(wave))


Selected wave: December 2021
Records: 603


In [ ]:
# --------------------------------------------------
# 2. Define labels for categorical variables new
# --------------------------------------------------

income_brackets = [
    "Under $10,000",
    "$10,000–$14,999",
    "$15,000–$19,999",
    "$20,000–$24,999",
    "$25,000–$29,999",
    "$30,000–$34,999",
    "$35,000–$39,999",
    "$40,000–$44,999",
    "$45,000–$49,999",
    "$50,000–$59,999",
    "$60,000–$74,999",
    "$75,000–$99,999",
    "$100,000–$124,999",
    "$125,000–$149,999",
    "$150,000–$174,999",
    "$175,000–$199,999",
    "$200,000–$224,999",
    "$225,000–$249,999",
    "$250,000 and above"
]

labels = {
    "SEX": {
        1: "Male",
        2: "Female"
    },

    "EDUCATION": {
        1: "Grade 1–12, no diploma",
        2: "High school diploma or GED",
        3: "Some college, no degree",
        4: "Associate degree or vocational certification",
        5: "Bachelor's degree",
        6: "Master's degree",
        7: "Professional degree",
        8: "Doctorate",
        99: "NA"
    },

    "MARRY": {
        1:	"Married/partner",
        2:	"Separated",
        3:	"Divorced",
        4:	"Widowed",
        5:	"Never married"
    },

    "HOMEOWN": {
        # Verify these codes against the December 2021 codebook
        1: "Owns or buying",
        2: "Rent",
        99: "NA"
    },

    "INCOME": {
        i: bracket
        for i, bracket in enumerate(income_brackets, start=1)
    }

}


In [ ]:
# --------------------------------------------------
# 4. Create a copy with selected variables and labels
# --------------------------------------------------

selected_variables = [
    "ID", "BIRTHM", "BIRTHY", "EDUC", "SEX", "MARRY",
    "NUMKID", "NUMADT", "INCOME", "HOMEOWN", "WT"
]

# Keep only the selected variables
persona_df = wave[selected_variables].copy()

# Apply mappings to categorical variables
for col, mapping in labels.items():
    if col in persona_df.columns:
        persona_df[col] = persona_df[col].map(mapping)

# Rename columns for readability
persona_df = persona_df.rename(columns={
    "ID": "Respondent_ID",
    "BIRTHM": "Birth_Month",
    "BIRTHY": "Birth_Year",
    "EDUC": "Education",
    "SEX": "Gender",
    "MARRY": "Marital_Status",
    "NUMKID": "Number_of_Children",
    "NUMADT": "Number_of_Adults",
    "INCOME": "Household_Income",
    "HOMEOWN": "Homeownership",
    "WT": "Survey_Weight"
})

# Inspect the result
print("Number of respondents:", len(persona_df))
print("Columns:", persona_df.columns.tolist())
display(persona_df.head())

In [ ]:
# --------------------------------------------------
# 3. Define demographic coding from the codebook
# --------------------------------------------------
labels = {
    "SEX": {
        1: "Male",
        2: "Female"
    },
    "REGION": {
        1: "West",
        2: "North Central",
        3: "Northeast",
        4: "South"
    },
    "MARRY": {
        1: "Married or partnered",
        3: "Divorced or separated",
        4: "Widowed",
        5: "Never married"
    },
    "EDUCATION": {
        1: "Grade 1–12, no diploma",
        2: "High school diploma or GED",
        3: "Some college, no degree",
        4: "Associate degree or vocational certification",
        5: "Bachelor's degree",
        6: "Master's degree",
        7: "Professional degree",
        8: "Doctorate"
    }
}

income_brackets = [
    "Under $10,000",
    "$10,000–$14,999",
    "$15,000–$19,999",
    "$20,000–$24,999",
    "$25,000–$29,999",
    "$30,000–$34,999",
    "$35,000–$39,999",
    "$40,000–$44,999",
    "$45,000–$49,999",
    "$50,000–$59,999",
    "$60,000–$74,999",
    "$75,000–$99,999",
    "$100,000–$124,999",
    "$125,000–$149,999",
    "$150,000–$174,999",
    "$175,000–$199,999",
    "$200,000–$224,999",
    "$225,000–$249,999",
    "$250,000 and above"
]

income_bins = [
    float("-inf"),
    10000, 15000, 20000, 25000, 30000,
    35000, 40000, 45000, 50000, 60000,
    75000, 100000, 125000, 150000, 175000,
    200000, 225000, 250000, float("inf")
]

df["INCOME_BRACKET"] = pd.cut(
    pd.to_numeric(df["INCOME"], errors="coerce"),
    bins=income_bins,
    labels=income_brackets,
    right=False
)
# --------------------------------------------------
# 4. Prepare demographic sampling frame
# Do not filter on GAS1PX1: that is the outcome
# --------------------------------------------------
persona_vars = [
    "AGE", "SEX", "REGION", "MARRY",
    "EDUCATION", "YTL5", "WT"
]

frame = wave[persona_vars].copy()

# Convert documented special/missing codes to NA
frame.loc[frame["AGE"] == 97, "AGE"] = np.nan
frame.loc[frame["EDUCATION"] == 99, "EDUCATION"] = np.nan

# Keep respondents with valid core demographics
core = ["SEX", "REGION", "EDUCATION", "YTL5"]
frame = frame.dropna(subset=core)

# Restrict codes to values defined by the codebook
frame = frame[
    frame["SEX"].isin([1, 2])
    & frame["REGION"].isin([1, 2, 3, 4])
    & frame["EDUCATION"].isin(range(1, 9))
    & frame["YTL5"].isin(range(1, 6))
].copy()

# Missing weights cannot be used for weighted sampling
frame = frame[
    frame["WT"].notna() & (frame["WT"] > 0)
].copy()

if len(frame) < 30:
    raise ValueError(
        f"Only {len(frame)} eligible records; cannot sample 30."
    )

# --------------------------------------------------
# 5. Draw 30 records using the survey weights
# --------------------------------------------------
sample = frame.sample(
    n=30,
    replace=False,
    weights="WT",
    random_state=202602
).copy()



In [4]:
# --------------------------------------------------
# 6. Translate values into persona characteristics
# --------------------------------------------------
personas = pd.DataFrame({
    "persona_id": [
        f"MI_202602_{i:03d}" for i in range(1, 31)
    ],
    "survey_wave": latest_wave,
    "age": sample["AGE"].values,
    "sex": sample["SEX"].map(labels["SEX"]).values,
    "region": sample["REGION"].map(labels["REGION"]).values,
    "marital_status": sample["MARRY"].map(
        labels["MARRY"]
    ).values,
    "education": sample["EDUCATION"].map(
        labels["EDUCATION"]
    ).values,
    "income_group": sample["YTL5"].map(
        labels["YTL5"]
    ).values
})

# Do not export respondent IDs, survey answers, or weights
output_path = "Michigan_personas_30.csv"
personas.to_csv(output_path, index=False)

print(personas.head(10).to_string(index=False))
print(f"\nSaved {len(personas)} personas to {output_path}")

   persona_id  survey_wave  age    sex    region        marital_status                                    education                      income_group
MI_202602_001       202602 76.0 Female      West         Never married                            Bachelor's degree  Lowest household income quintile
MI_202602_002       202602 37.0   Male Northeast Divorced or separated                   High school diploma or GED  Lowest household income quintile
MI_202602_003       202602 79.0   Male     South               Widowed                   High school diploma or GED  Lowest household income quintile
MI_202602_004       202602 48.0   Male      West  Married or partnered Associate degree or vocational certification Highest household income quintile
MI_202602_005       202602 82.0   Male     South  Married or partnered                              Master's degree  Middle household income quintile
MI_202602_006       202602 38.0   Male Northeast         Never married                              